In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.append(str(PROJECT_ROOT))

from src.data_factory.pdf_loader import PDFLoader
from src.data_factory.cleaner import PDFCleaner
from src.data_factory.chunker import PDFChunker
from src.utils.file_io import save_json
from src.utils.logger import setup_logger

# Define paths

In [2]:
logger = setup_logger()

PDF_PATH = PROJECT_ROOT / "data" / "raw" / "uber_2024_annual_report.pdf"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "chunks.json"

PDF_PATH, OUTPUT_PATH

(WindowsPath('c:/Users/Dunith Munasinghe/Desktop/ledger-mind/Annual-Report-Reading-RAG-Finetunning/data/raw/uber_2024_annual_report.pdf'),
 WindowsPath('c:/Users/Dunith Munasinghe/Desktop/ledger-mind/Annual-Report-Reading-RAG-Finetunning/data/processed/chunks.json'))

# Load PDF page by page

In [3]:
loader = PDFLoader(PDF_PATH)

raw_pages = loader.load_pages()

logger.info(f"Total pages loaded: {len(raw_pages)}")

raw_pages[:2]

Checking weak pages with pdfplumber fallback: 100%|██████████| 142/142 [00:00<00:00, 190.63it/s]

2026-05-23 22:56:55 | INFO | __main__:<module>:5 | Total pages loaded: 142


[{'page_number': 1,
  'text': 'On Our Way\n2024 ANNUAL REPORT',
  'source': 'pymupdf'},
 {'page_number': 2,
  'text': 'Uber’s Mission\nWe reimagine the way the world moves for the better\nWe are Uber. The go-getters. The kind of people who are relentless about our \nmission to help people go anywhere and get anything and earn their way. \nMovement is what we power. It’s our lifeblood. It runs through our veins. It’s \nwhat gets us out of bed each morning. It pushes us to constantly reimagine \nhow we can move better. For you. For all the places you want to go. For all the \nthings you want to get. For all the ways you want to earn. Across the entire \nworld. In real time. At the incredible speed of now.',
  'source': 'pymupdf'}]

# Quick extraction quality check

In [4]:
page_quality = pd.DataFrame(
    [
        {
            "page_number": page["page_number"],
            "source": page["source"],
            "char_count": len(page["text"]),
            "preview": page["text"][:120],
        }
        for page in raw_pages
    ]
)

page_quality.head(10)

,page_number,source,char_count,preview
0,1,pymupdf,29,On Our Way\n2024 ANNUAL REPORT
1,2,pymupdf,585,Uber’s Mission\nWe reimagine the way the world...
2,3,pymupdf,2660,UNITED STATES \nSECURITIES AND EXCHANGE COMMIS...
3,4,pymupdf,2253,Large accelerated filer \n☒ \n \nAccelerated ...
4,5,pymupdf,1537,"1 \nUBER TECHNOLOGIES, INC. \nTABLE OF CONTENT..."
5,6,pymupdf,4458,2 \nSPECIAL NOTE REGARDING FORWARD-LOOKING STA...
6,7,pymupdf,2544,3 \nActual events or results may differ from t...
7,8,pymupdf,5042,4 \nPART I \nITEM 1. BUSINESS \nOverview \nUbe...
8,9,pymupdf,5288,5 \nPlatform Synergies \nOur Platform \nThe fo...
9,10,pymupdf,6520,6 \n• \nMobility. Our Mobility offering compet...


# Clean pages

In [5]:
cleaner = PDFCleaner(raw_pages)

cleaned_pages = cleaner.clean_pages()

logger.info(f"Total pages cleaned: {len(cleaned_pages)}")

cleaned_pages[:2]

2026-05-23 23:03:03 | INFO | __main__:<module>:5 | Total pages cleaned: 142


[{'page_number': 1,
  'section_title': 'On Our Way',
  'text': 'On Our Way 2024 ANNUAL REPORT',
  'source': 'pymupdf'},
 {'page_number': 2,
  'section_title': 'Uber’s Mission',
  'text': 'Uber’s Mission We reimagine the way the world moves for the better We are Uber. The go-getters. The kind of people who are relentless about our mission to help people go anywhere and get anything and earn their way. Movement is what we power. It’s our lifeblood. It runs through our veins. It’s what gets us out of bed each morning. It pushes us to constantly reimagine how we can move better. For you. For all the places you want to go. For all the things you want to get. For all the ways you want to earn. Across the entire world. In real time. At the incredible speed of now.',
  'source': 'pymupdf'}]

# Check cleaned text quality

In [6]:
cleaned_quality = pd.DataFrame(
    [
        {
            "page_number": page["page_number"],
            "section_title": page["section_title"],
            "char_count": len(page["text"]),
            "preview": page["text"][:150],
        }
        for page in cleaned_pages
    ]
)

cleaned_quality.head(10)

,page_number,section_title,char_count,preview
0,1,On Our Way,29,On Our Way 2024 ANNUAL REPORT
1,2,Uber’s Mission,579,Uber’s Mission We reimagine the way the world ...
2,3,UNITED STATES,2347,UNITED STATES SECURITIES AND EXCHANGE COMMISSI...
3,4,Large accelerated filer,2186,Large accelerated filer Accelerated filer Non-...
4,5,"UBER TECHNOLOGIES, INC.",1318,"UBER TECHNOLOGIES, INC. TABLE OF CONTENTS Page..."
5,6,SPECIAL NOTE REGARDING FORWARD-LOOKING STATEMENTS,4331,SPECIAL NOTE REGARDING FORWARD-LOOKING STATEME...
6,7,could differ materially from those described i...,2521,Actual events or results may differ from those...
7,8,PART I,4990,PART I ITEM 1. BUSINESS Overview Uber Technolo...
8,9,Platform Synergies,5219,Platform Synergies Our Platform The foundation...
9,10,"ridesharing companies for Drivers and Riders, ...",6454,Mobility. Our Mobility offering competes with ...


# Chunk into 1500-character chunks

In [7]:
chunker = PDFChunker(
    document_id="uber_2024",
    chunk_size=1500,
    chunk_overlap=150,
)

chunks = chunker.chunk_pages(cleaned_pages)

logger.info(f"Total chunks created: {len(chunks)}")

chunks[:3]

2026-05-23 23:03:58 | INFO | __main__:<module>:9 | Total chunks created: 522


[{'chunk_id': 'uber_2024_p2_c001',
  'page_start': 2,
  'page_end': 2,
  'section_title': 'Uber’s Mission',
  'text': 'Uber’s Mission We reimagine the way the world moves for the better We are Uber. The go-getters. The kind of people who are relentless about our mission to help people go anywhere and get anything and earn their way. Movement is what we power. It’s our lifeblood. It runs through our veins. It’s what gets us out of bed each morning. It pushes us to constantly reimagine how we can move better. For you. For all the places you want to go. For all the things you want to get. For all the ways you want to earn. Across the entire world. In real time. At the incredible speed of now.',
  'char_count': 579},
 {'chunk_id': 'uber_2024_p3_c002',
  'page_start': 3,
  'page_end': 3,
  'section_title': 'UNITED STATES',
  'text': 'UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 FORM 10-K (Mark One) ☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES 

# Validate chunk format

In [8]:
chunk_df = pd.DataFrame(chunks)

chunk_df.head()

,chunk_id,page_start,page_end,section_title,text,char_count
0,uber_2024_p2_c001,2,2,Uber’s Mission,Uber’s Mission We reimagine the way the world ...,579
1,uber_2024_p3_c002,3,3,UNITED STATES,UNITED STATES SECURITIES AND EXCHANGE COMMISSI...,1311
2,uber_2024_p3_c003,3,3,UNITED STATES,. Yes ☒ No ☐ Indicate by check mark whether th...,1177
3,uber_2024_p4_c004,4,4,Large accelerated filer,Large accelerated filer Accelerated filer Non-...,1315
4,uber_2024_p4_c005,4,4,Large accelerated filer,. Indicate by check mark whether the registran...,984


# Check chunk length distribution

In [9]:
chunk_df["char_count"].describe()

count     522.000000
mean     1207.304598
std       348.013156
min        22.000000
25%      1129.500000
50%      1354.500000
75%      1440.750000
max      1500.000000
Name: char_count, dtype: float64

In [10]:
save_json(chunks, OUTPUT_PATH)

logger.info(f"Chunks saved to: {OUTPUT_PATH}")

2026-05-23 23:05:46 | INFO | __main__:<module>:3 | Chunks saved to: c:\Users\Dunith Munasinghe\Desktop\ledger-mind\Annual-Report-Reading-RAG-Finetunning\data\processed\chunks.json


In [11]:
import json

with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    saved_chunks = json.load(f)

print("Total saved chunks:", len(saved_chunks))
print(saved_chunks[0])

Total saved chunks: 522
{'chunk_id': 'uber_2024_p2_c001', 'page_start': 2, 'page_end': 2, 'section_title': 'Uber’s Mission', 'text': 'Uber’s Mission We reimagine the way the world moves for the better We are Uber. The go-getters. The kind of people who are relentless about our mission to help people go anywhere and get anything and earn their way. Movement is what we power. It’s our lifeblood. It runs through our veins. It’s what gets us out of bed each morning. It pushes us to constantly reimagine how we can move better. For you. For all the places you want to go. For all the things you want to get. For all the ways you want to earn. Across the entire world. In real time. At the incredible speed of now.', 'char_count': 579}
